# Super-resolution exploration

This notebook inspects the data and compares evaluation outputs.

In [ ]:
from pathlib import Path
import pandas as pd
from utils.config import load_config

cfg = load_config('configs/evaluation.yaml')
print(cfg.evaluation.to_dict())

In [ ]:
from collections import Counter
from sr_datasets.sr_dataset import SRDataset

dataset = SRDataset(cfg, 'test', 'test')
city_counts = Counter(dataset.rows[i]['patch_name'].split('_y', 1)[0].rstrip('0123456789') for i in range(len(dataset.rows)))
city_counts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, index in zip(axes, [0, 1, 2]):
    sample = dataset[index]
    axis.imshow(sample['hr'].permute(1, 2, 0))
    axis.set_title(sample['name'])
    axis.axis('off')
plt.show()

Run TensorBoard from the project root:

`tensorboard --logdir runs`

In [ ]:
results_path = Path('results/metrics/results.csv')
results = pd.read_csv(results_path) if results_path.is_file() else pd.DataFrame()
results

In [ ]:
import torch
from inference.super_resolve import load_model_for_inference, super_resolve_tensor

checkpoint = Path('checkpoints/srgan/srgan_gpu/best_generator.pth')
model = load_model_for_inference('srgan', checkpoint, 'configs/srgan.yaml', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
sr = super_resolve_tensor(model, dataset[0]['lr'], 128, 16, next(model.parameters()).device, False)
sr.shape